In [ ]:
!pip install mlflow torch torchvision scikit-learn pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.

In [ ]:
import mlflow
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import time, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# MLflow tracks to a local SQLite file — 100% free, no server needed
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("image_classifier_experiments")

Using: cuda


2026/09/18 06:49:37 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/18 06:49:37 INFO mlflow.store.db.utils: Updating database tables
2026/09/18 06:49:39 INFO mlflow.tracking.fluent: Experiment with name 'image_classifier_experiments' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1789714179910, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789714179910, lifecycle_stage='active', name='image_classifier_experiments', tags={}, trace_location=None, workspace='default'>

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
full_test  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Subset for fast Colab training
train_set = Subset(full_train, range(8000))
test_set  = Subset(full_test, range(2000))

classes = full_train.classes

100%|██████████| 170M/170M [32:41<00:00, 86.9kB/s]


In [4]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

In [5]:
def train_and_log(lr, epochs, batch_size, run_name):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_set, batch_size=batch_size)

    model = SimpleCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"lr": lr, "epochs": epochs, "batch_size": batch_size})

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            for imgs, labels in train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                out = model(imgs)
                loss = criterion(out, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            avg_loss = running_loss / len(train_loader)
            mlflow.log_metric("train_loss", avg_loss, step=epoch)

        # Evaluate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        mlflow.log_metric("val_accuracy", val_acc)

        # Save model artifact for this run
        model_path = f"model_{run_name}.pth"
        torch.save(model.state_dict(), model_path)
        mlflow.log_artifact(model_path)

        print(f"{run_name}: lr={lr}, epochs={epochs}, batch={batch_size} → val_acc={val_acc:.4f}")
        return val_acc, model, model_path

In [6]:
configs = [
    {"lr": 0.001, "epochs": 3, "batch_size": 64, "run_name": "run_lr001_bs64"},
    {"lr": 0.0005, "epochs": 3, "batch_size": 64, "run_name": "run_lr0005_bs64"},
    {"lr": 0.001, "epochs": 5, "batch_size": 32, "run_name": "run_lr001_bs32_ep5"},
]

results = []
for cfg in configs:
    val_acc, model, model_path = train_and_log(**cfg)
    results.append({"run_name": cfg["run_name"], "val_acc": val_acc,
                     "model": model, "model_path": model_path, "config": cfg})

best = max(results, key=lambda r: r["val_acc"])
print("\nBest run:", best["run_name"], "val_acc:", best["val_acc"])

run_lr001_bs64: lr=0.001, epochs=3, batch=64 → val_acc=0.5530
run_lr0005_bs64: lr=0.0005, epochs=3, batch=64 → val_acc=0.5145
run_lr001_bs32_ep5: lr=0.001, epochs=5, batch=32 → val_acc=0.5905

Best run: run_lr001_bs32_ep5 val_acc: 0.5905


In [ ]:
import pandas as pd

runs_df = mlflow.search_runs(experiment_names=["image_classifier_experiments"])
runs_df = runs_df[["tags.mlflow.runName", "params.lr", "params.epochs",
                    "params.batch_size", "metrics.val_accuracy", "metrics.train_loss"]]
runs_df.columns = ["run_name", "lr", "epochs", "batch_size", "val_accuracy", "train_loss"]
runs_df = runs_df.sort_values("val_accuracy", ascending=False)
runs_df.to_csv("runs_summary.csv", index=False)

# Save ONLY the best model for deployment
torch.save(best["model"].state_dict(), "best_model.pth")

print(runs_df)
print("\nSaved: runs_summary.csv, best_model.pth")

In [9]:
from google.colab import files
files.download("runs_summary.csv")
files.download("best_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>